# Linear Regression: Zero to One
### Project: House price predictor on the California Housing dataset

This notebook is built step by step. **Step 1 (this section): Theory in plain English.**
No math background needed. Every symbol is defined the first time it appears.

## 1. Theory

### 1.1 What is regression?
**Regression** means predicting a *number* from other numbers.

- Predict a house's **price** (a number) from its **size, age, and location** (more numbers).
- The thing we predict is the **target** (also called the label), written $y$.
- The things we use to predict are the **features** (also called inputs), written $x$.
- **Training** means showing the computer many houses where we already know the price, so it learns the pattern.

If the target were a category ("cheap" / "expensive") it would be *classification*. A number means *regression*.

### 1.2 The line equation
You may remember from school: $y = mx + b$. Machine learning uses different letter names:

$$\hat{y} = w x + b$$

| Symbol | Name | Plain meaning |
|---|---|---|
| $\hat{y}$ ("y-hat") | prediction | what our model guesses |
| $y$ | true value | the real price |
| $x$ | feature | e.g. house size |
| $w$ | **weight** (slope) | how much the price changes when $x$ goes up by 1 |
| $b$ | **bias** (intercept) | the price when $x = 0$; the line's starting height |

Example: if $w = 50$ and $b = 20$ (in thousands of dollars), then a house of size 10 is predicted as $50 \cdot 10 + 20 = 520$.

With many features (size, age, rooms...) we just add more weights:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b$$

Here $d$ is the number of features. **Training means finding the best $w$'s and $b$.**

### 1.3 What does "best" mean? Error and residuals
For one house, the **residual** (error) is the gap between truth and guess:

$$\text{residual} = y - \hat{y}$$

A good line has small residuals across *all* houses. But we need ONE number that summarizes "how wrong is the line overall". That number is the **cost function**.

### 1.4 Why squared error?
The most common choice is to square each residual and average them. This is the **Mean Squared Error (MSE)**:

$$J(w, b) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2$$

Here $n$ is the number of houses, $i$ labels one house, and $\sum$ means "add up over all houses". $J$ is the **cost function** (also called loss function): a score where **lower = better**.

**Why square instead of just adding residuals?**
1. **Signs cancel.** A residual of +10 and one of -10 would sum to 0, which looks perfect but isn't. Squaring makes everything positive.
2. **Smooth.** The squared curve has no sharp corners, so calculus (and gradient descent) works nicely. Absolute value has a sharp corner at 0.
3. **Big errors are punished much more.** Being off by 2 costs $2^2 = 4$. Being off by 10 costs $10^2 = 100$. Five times the error costs **25 times** the penalty. The line is therefore pulled hard to avoid large misses. This is good if big misses are truly bad, and bad if your data has a few weird outliers, because they dominate the fit.

The code cell below shows this visually.

In [ ]:
# --- Setup: import the libraries we need for the theory demos ---
import numpy as np                 # NumPy: fast math on arrays of numbers
import matplotlib.pyplot as plt    # Matplotlib: plotting library

# Make plots a bit larger and easier to read
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
# --- Demo: how squared error punishes big mistakes more than absolute error ---

# A range of possible residuals (errors) from -10 to +10, 200 evenly spaced points
residuals = np.linspace(-10, 10, 200)

# Absolute error: just the size of the error, ignoring sign (grows in a straight line)
abs_error = np.abs(residuals)

# Squared error: error multiplied by itself (grows as a curve, faster and faster)
sq_error = residuals ** 2

# Plot both penalties against the size of the error
plt.plot(residuals, abs_error, label="Absolute error |e|")
plt.plot(residuals, sq_error, label="Squared error e^2")
plt.xlabel("Residual (y - y_hat)")
plt.ylabel("Penalty")
plt.title("Squared error punishes big mistakes far more")
plt.legend()
plt.show()

# Numeric version: compare penalties for a small and a big error
for e in [2, 10]:
    print(f"error = {e:>2}  ->  absolute penalty = {abs(e):>3},  squared penalty = {e**2:>3}")

**What the plot tells you:** near 0 the squared curve is *flatter* than the absolute line (small errors barely matter), but beyond about 1 it shoots upward. An error of 10 is only 5x worse than an error of 2 in absolute terms, but **25x worse** in squared terms.

### 1.5 The cost function as a landscape
Imagine only one unknown weight $w$ (ignore $b$). For each possible $w$ we can compute the cost $J(w)$. Plotting $J$ against $w$ gives a **bowl-shaped curve** (a parabola). The bottom of the bowl is the best $w$. With two unknowns the bowl becomes a 3D valley, and so on in higher dimensions. For linear regression with squared error the bowl always has **exactly one bottom** (the technical word is *convex*), so there are no tricky local traps.

The demo below builds a tiny dataset and draws this bowl.

In [ ]:
# --- Demo: the cost "bowl" for a tiny dataset with one feature and no bias ---

# Tiny dataset: the true relationship is y = 3 * x exactly
x = np.array([1.0, 2.0, 3.0, 4.0])   # features
y = 3.0 * x                           # targets

# Try many candidate weights from -2 to 8
w_values = np.linspace(-2, 8, 200)

# For each candidate w: predict (w * x), compute residuals, square them, average them
costs = [np.mean((y - w * x) ** 2) for w in w_values]

# Find the w with the smallest cost
best_w = w_values[np.argmin(costs)]

plt.plot(w_values, costs)
plt.axvline(best_w, color="red", linestyle="--", label=f"best w ~ {best_w:.2f}")
plt.xlabel("weight w")
plt.ylabel("cost J(w) (MSE)")
plt.title("Cost function is a bowl: one lowest point")
plt.legend()
plt.show()

### 1.6 Gradient descent: walking down the bowl
How does a computer find the bottom? **Gradient descent** is like standing on a foggy hillside and wanting to reach the valley:

1. Feel which direction the ground slopes **downhill** right where you stand.
2. Take a small step that way.
3. Repeat until the ground is flat (you're at the bottom).

The mathematical names:

- **Gradient** $\nabla J$: the slope of the cost. It points *uphill*, so we step in the *opposite* direction. (It is computed using derivatives, which measure "how fast does $J$ change if I nudge $w$ a tiny bit".)
- **Learning rate** $\eta$ ("eta"): the step size. Too small = painfully slow. Too big = you leap over the valley and may bounce out of control.
- **Iteration** (or epoch, when it passes over all data once): one repeat of the loop.

The update rule, applied to all weights at once:

$$w \leftarrow w - \eta \, \nabla J(w)$$

The arrow $\leftarrow$ means "replace $w$ with the value on the right".

### 1.7 The normal equation: skipping the walk
Because the bowl has a single bottom, we can *calculate* where it is directly: the bottom is where the slope is zero. Solving that gives a one-line formula, the **normal equation**. Below is the full derivation.

#### Setup in matrix form
Stack all houses into a **matrix** $X$ (a grid of numbers) with $n$ rows (houses) and $d+1$ columns (features). The extra column is all 1s so the bias $b$ can be treated as just another weight. Put all weights in a **vector** $w$ (a list of numbers, now including $b$). Then all predictions at once are:

$$\hat{y} = Xw$$

Notation: $X^T$ is the **transpose** (rows and columns swapped). $X^{-1}$ is the **inverse** (the matrix that "undoes" $X$, like dividing). $\|v\|^2 = v^T v$ means "sum of squares of the entries of $v$".

#### Step 1: write the cost in matrix form
$$J(w) = \frac{1}{n}(y - Xw)^T (y - Xw)$$
*Why:* $v^T v$ squares each entry and adds them up, which is exactly the sum of squared residuals.

#### Step 2: expand the brackets
$$J(w) = \frac{1}{n}\left(y^Ty - y^TXw - w^TX^Ty + w^TX^TXw\right)$$
*Why:* same as $(a-b)(a-b)$ but with matrices; we use $(Xw)^T = w^TX^T$.

#### Step 3: combine the two middle terms
$y^TXw$ is a single number, and a single number equals its own transpose, so $y^TXw = w^TX^Ty$:
$$J(w) = \frac{1}{n}\left(y^Ty - 2w^TX^Ty + w^TX^TXw\right)$$

#### Step 4: take the gradient (the slope with respect to $w$)
Two standard rules: the gradient of $w^Ta$ is $a$, and the gradient of $w^TAw$ is $2Aw$ (for symmetric $A$, and $X^TX$ is symmetric). The first term $y^Ty$ does not contain $w$, so its slope is 0:
$$\nabla J(w) = \frac{1}{n}\left(-2X^Ty + 2X^TXw\right)$$

#### Step 5: set the slope to zero (the bottom of the bowl)
$$-2X^Ty + 2X^TXw = 0 \;\Rightarrow\; X^TXw = X^Ty$$
*Why:* move one term across and divide both sides by 2. This is the **normal equation**.

#### Step 6: solve for $w$
Multiply both sides on the left by $(X^TX)^{-1}$:
$$\boxed{\,w = (X^TX)^{-1}X^Ty\,}$$

**Caveats you'll meet later:**
- It needs $X^TX$ to be invertible. If two features are copies/combinations of each other (**multicollinearity**) it breaks or becomes unstable.
- Inverting a big matrix is slow when there are very many features, which is when gradient descent wins.
- The same slope formula gives the gradient descent update: $\nabla J(w) = \frac{2}{n}X^T(Xw - y)$.

### 1.8 Step 1 summary
| Concept | One-line takeaway |
|---|---|
| Regression | Predict a number from features |
| Line | $\hat{y} = w_1x_1 + \dots + b$; training finds $w, b$ |
| Squared error | Positive, smooth, punishes big misses (error x5 means penalty x25) |
| Cost function | One score for the whole line, lower is better, bowl-shaped |
| Gradient descent | Repeated small downhill steps, controlled by the learning rate |
| Normal equation | Jump straight to the bottom: $w = (X^TX)^{-1}X^Ty$ |

**Next up (Step 2):** exploring the California Housing data with plots.

## 2. Exploratory Data Analysis (EDA)

**EDA** means *looking at your data before modeling*. We check what each column looks like, how columns relate, and what is weird. Models only learn what the data shows, so surprises found here save hours later.

### 2.1 Load the data
The **California Housing** dataset has 20,640 *block groups* (a block group is a small neighborhood of roughly 1,000 people from the 1990 census). One row = one block group. Columns:

| Column | Meaning |
|---|---|
| `MedInc` | median income in the block group (units: $10,000) |
| `HouseAge` | median age of houses (years) |
| `AveRooms` | average rooms per household |
| `AveBedrms` | average bedrooms per household |
| `Population` | people living in the block group |
| `AveOccup` | average people per household |
| `Latitude`, `Longitude` | location |
| **`MedHouseVal`** | **target**: median house value (units: $100,000) |

In [ ]:
# --- Imports for EDA ---
import numpy as np                       # numerical arrays
import pandas as pd                      # tables (DataFrames)
import matplotlib.pyplot as plt          # plotting
import seaborn as sns                    # nicer statistical plots built on matplotlib
from sklearn.datasets import fetch_california_housing   # dataset loader

sns.set_theme(style="whitegrid")         # clean plot style with light grid lines

In [ ]:
# --- Load the dataset as a pandas DataFrame ---
# as_frame=True gives us a DataFrame instead of raw NumPy arrays
# (the first run downloads the data; it is then cached on your machine)
housing = fetch_california_housing(as_frame=True)

# housing.frame contains the 8 feature columns plus the target column 'MedHouseVal'
df = housing.frame

# Show the first 5 rows to get a feel for the data
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
# --- Basic health check ---
# .info() lists each column, its data type, and how many values are non-missing
df.info()

# Count missing values per column (we expect all zeros for this dataset)
print("\nMissing values per column:\n", df.isna().sum())

# .describe() gives count, mean, std (spread), min, quartiles (25/50/75%), max
# Look at 'max' vs '75%' to spot columns with extreme values
df.describe().T

**What this tells you:** there are no missing values (good). But look at `max` vs `75%`: `AveRooms` max is about 140 and `AveOccup` max is over 1,000, while their typical values are around 5 and 3. Those are almost certainly odd blocks (e.g. vacation areas or group housing). We'll see them clearly in the plots below.

### 2.2 Distributions (histograms)
A **histogram** splits a column's range into bins and counts how many rows fall in each. It shows the *shape* of the data: symmetric, skewed (a long tail on one side), or having spikes.

In [ ]:
# --- Histograms of every column ---
# df.hist draws one histogram per column; bins=40 means 40 bars each
axes = df.hist(bins=40, figsize=(14, 9), color="steelblue", edgecolor="white")

plt.suptitle("Distribution of every column", y=1.02, fontsize=14)  # overall title
plt.tight_layout()                                                  # avoid overlapping labels
plt.show()

**What each histogram tells you:**
- `MedInc`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`: **right-skewed** (most values small, a long tail of large ones). Linear regression prefers roughly symmetric data, so later we may apply a log transform.
- `HouseAge`: fairly flat with a spike at 52, meaning ages above 52 were lumped into 52 (**capping**).
- `MedHouseVal` (target): a tall spike at the far right (5.0 = $500,000). The data was **capped** there, so true values above $500k were recorded as 5.0. A line can't learn the real prices of those houses.
- `Latitude`/`Longitude`: two clusters, roughly the Bay Area and Los Angeles.

In [ ]:
# --- Quantify the target cap ---
# Count how many block groups have the maximum price value
cap_value = df["MedHouseVal"].max()                       # the largest target value
n_capped = (df["MedHouseVal"] == cap_value).sum()         # how many rows equal it
print(f"Max value = {cap_value}  ->  {n_capped} rows ({n_capped/len(df):.1%}) sit exactly at the cap")

### 2.3 Correlations
**Correlation** (Pearson's $r$) is a number from -1 to +1 describing how strongly two columns move together along a straight line:
- $+1$: when one goes up, the other always goes up (perfect straight line).
- $0$: no linear relationship.
- $-1$: when one goes up, the other goes down.

A **heatmap** colors a table of all pairwise correlations. **Caution:** correlation shows association, not cause, and it only detects *straight-line* relationships.

In [ ]:
# --- Correlation heatmap ---
corr = df.corr()                                    # table of pairwise correlations

plt.figure(figsize=(9, 7))
# annot=True prints the numbers; cmap 'coolwarm' = blue (negative) to red (positive)
# vmin/vmax fix the color scale to the full -1..1 range so colors are comparable
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation between columns")
plt.show()

# Rank features by how correlated they are with the target
print(corr["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False))

**What the heatmap tells you:**
- `MedInc` has the strongest link to price (about +0.69): richer neighborhoods have pricier homes. It will be our most important feature.
- `AveRooms` and `AveBedrms` are strongly correlated with *each other*. Two features carrying similar information is **multicollinearity**, which we'll test with VIF in Step 5.
- `Latitude` and `Longitude` are strongly (negatively) correlated with each other because California runs diagonally.

In [ ]:
# --- Scatter plots: feature vs target ---
# Use a random sample of 3000 points; plotting all 20,640 would just be a blob
sample = df.sample(3000, random_state=42)    # random_state makes the sample reproducible

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["MedInc", "HouseAge", "AveRooms"]):
    ax.scatter(sample[col], sample["MedHouseVal"], alpha=0.3, s=10)   # alpha = transparency
    ax.set_xlabel(col)
    ax.set_ylabel("MedHouseVal")
    ax.set_title(f"{col} vs price")
plt.tight_layout()
plt.show()

**What the scatter plots tell you:**
- `MedInc` vs price: a clear upward trend (linear regression suits this), a horizontal row of dots at 5.0 (the cap), and the spread gets wider as income grows. That fanning-out is the heteroscedasticity we test in Step 5.
- `HouseAge` vs price: almost a cloud, so a weak linear relationship.
- `AveRooms` vs price: a few extreme x values squash everything else into a thin band, which shows the outlier problem.

### 2.4 Outliers
An **outlier** is a value far from the rest. Outliers matter for regression because squared error lets them pull the line towards them. A **box plot** shows the middle 50% of data as a box; dots beyond the "whiskers" are flagged outliers. The **IQR rule** (IQR = Q3 - Q1, the box height) flags points below $Q1 - 1.5\cdot IQR$ or above $Q3 + 1.5\cdot IQR$.

In [ ]:
# --- Box plots of the skewed columns (separate axes since scales differ wildly) ---
cols = ["AveRooms", "AveBedrms", "Population", "AveOccup"]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, cols):
    sns.boxplot(y=df[col], ax=ax, color="lightcoral")   # one box per column
    ax.set_title(col)
plt.tight_layout()
plt.show()

# --- Count outliers per column using the IQR rule ---
def iqr_outlier_count(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)       # 25th and 75th percentiles
    iqr = q3 - q1                                      # the box height
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr      # whisker limits
    return ((s < lower) | (s > upper)).sum()           # count values outside the limits

print("IQR-rule outlier counts:")
for col in df.columns:
    print(f"  {col:12s} {iqr_outlier_count(df[col]):5d}")

**What the box plots tell you:** the boxes are squashed flat near the bottom because a handful of extreme values (e.g. an `AveRooms` of 140) stretch the axis. Those points are real-world oddities, not typical houses.

**What to do (we'll use this later):** options are (1) remove clearly implausible rows, (2) **clip** (cap) values at a percentile, (3) log-transform skewed columns, (4) leave them and use a robust model. We'll decide when we model.

### 2.5 Geography
Latitude and longitude plotted as x/y make a rough map of California. Coloring by price shows location effects that a straight-line model on the raw numbers can't fully capture.

In [ ]:
# --- Map-like scatter: longitude (x) vs latitude (y), colored by price ---
plt.figure(figsize=(8, 7))
sc = plt.scatter(df["Longitude"], df["Latitude"],
                 c=df["MedHouseVal"],        # color each point by price
                 cmap="viridis", s=4, alpha=0.5)
plt.colorbar(sc, label="MedHouseVal ($100k)")  # color legend
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("House prices across California")
plt.show()

**What the map tells you:** the coastline is visible, and the expensive (bright) areas hug the coast near the Bay Area and Los Angeles, while inland is cheaper. Price depends on location in a *non-linear* way, so a plain linear model will have some limits here. That's a useful thing to know before we model.

### 2.6 Step 2 summary
| Finding | Consequence for modeling |
|---|---|
| No missing values | No imputation needed |
| Skewed features + extreme outliers | Consider log transform / clipping |
| Target capped at 5.0 | Model can't learn prices above $500k; expect errors there |
| `MedInc` is the top predictor | Expect a large coefficient |
| `AveRooms` / `AveBedrms` correlated | Check multicollinearity (VIF) later |
| Variance grows with income | Check heteroscedasticity later |

**Next up (Step 3):** building linear regression from scratch with NumPy.

## 3. Linear Regression From Scratch (NumPy only)

Now we turn the theory into code. We implement the **normal equation** and **gradient descent** ourselves using only NumPy, then check they agree with scikit-learn.

### 3.1 Prepare the data
Two preparation ideas, defined:

- **Train/test split:** we hide a portion of the data (the **test set**) during training and use it afterwards to measure how well the model handles *unseen* houses. Scoring on training data would be like grading a student on the exact questions they memorized. We use 80% to train, 20% to test.
- **Standardization (scaling):** subtract each column's mean and divide by its standard deviation (**std**, the typical spread), so every feature has mean 0 and std 1. This makes gradient descent converge faster and makes coefficients comparable. **Important:** compute the mean/std from the *training* data only and reuse them on the test data. Using test data to compute them would be **data leakage** (the test set secretly influences training).

In [ ]:
# --- Split and standardize (using NumPy for the maths) ---
from sklearn.model_selection import train_test_split   # only used for the random split

# Separate features (everything except the target) from the target column
X = df.drop(columns="MedHouseVal").values     # shape (20640, 8): NumPy array of features
y = df["MedHouseVal"].values                  # shape (20640,):  NumPy array of prices
feature_names = list(df.drop(columns="MedHouseVal").columns)   # remember column names

# 80% train / 20% test; random_state=42 makes the split identical every run
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Mean and std computed from TRAINING data only (axis=0 -> one value per column)
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

# Standardize: subtract mean, divide by std. Test uses the TRAIN mu and sigma.
X_train_s = (X_train - mu) / sigma
X_test_s = (X_test - mu) / sigma

print("Train shape:", X_train_s.shape, " Test shape:", X_test_s.shape)
print("Train feature means (should be ~0):", np.round(X_train_s.mean(axis=0), 3))
print("Train feature stds  (should be ~1):", np.round(X_train_s.std(axis=0), 3))

### 3.2 (a) Normal equation
From Step 1: $w = (X^TX)^{-1}X^Ty$, where $X$ has an extra column of 1s for the bias.

Implementation note: instead of literally computing the inverse (slow and numerically fragile), we rearrange to $X^TXw = X^Ty$ and ask NumPy to **solve** that system with `np.linalg.solve`, which is the better practice. We also show the literal inverse version to prove they agree.

In [ ]:
# --- Normal equation from scratch ---
def add_bias_column(X):
    # Put a column of ones on the left so weight[0] acts as the bias b
    ones = np.ones((X.shape[0], 1))            # n rows, 1 column of 1s
    return np.hstack([ones, X])                # glue columns side by side

def normal_equation(X, y):
    Xb = add_bias_column(X)                    # shape (n, d+1)
    A = Xb.T @ Xb                              # X^T X  (@ is matrix multiplication)
    b = Xb.T @ y                               # X^T y
    return np.linalg.solve(A, b)               # solve A w = b for w

# Fit on the standardized training data
w_normal = normal_equation(X_train_s, y_train)

# Same answer via the literal formula (X^T X)^-1 X^T y, for comparison
Xb_train = add_bias_column(X_train_s)
w_inverse = np.linalg.inv(Xb_train.T @ Xb_train) @ Xb_train.T @ y_train

print("Max difference solve vs inverse:", np.abs(w_normal - w_inverse).max())

# Display the learned weights next to feature names
print("\nbias (intercept):", round(w_normal[0], 4))
for name, w in zip(feature_names, w_normal[1:]):
    print(f"{name:12s} {w:8.4f}")

**Reading the weights:** features are standardized, so each weight means "price change (in $100k) when this feature rises by one standard deviation, holding the others fixed". The bias equals the average training price (because the features are centered). Notice `Latitude` and `Longitude` have large negative weights and `MedInc` a large positive one.

### 3.3 (b) Gradient descent
From Step 1: the slope of the cost is $\nabla J(w) = \frac{2}{n}X^T(Xw - y)$, and we repeat $w \leftarrow w - \eta\nabla J(w)$.

Recipe per iteration: (1) predict with the current $w$, (2) compute the gradient, (3) step downhill, (4) record the cost so we can plot the learning curve.

In [ ]:
# --- Gradient descent from scratch ---
def gradient_descent(X, y, lr=0.1, n_iters=2000):
    Xb = add_bias_column(X)                    # add bias column of ones
    n, d = Xb.shape                            # n houses, d weights (incl. bias)
    w = np.zeros(d)                            # start with all weights = 0
    losses = []                                # store the cost at each iteration

    for _ in range(n_iters):
        preds = Xb @ w                         # current predictions
        errors = preds - y                     # (prediction - truth) for each house
        losses.append(np.mean(errors ** 2))    # MSE cost J(w) at this step
        grad = (2 / n) * (Xb.T @ errors)       # gradient of MSE w.r.t. every weight
        w = w - lr * grad                      # step opposite to the gradient

    return w, losses

# Run it: learning rate 0.1, 2000 iterations
w_gd, losses = gradient_descent(X_train_s, y_train, lr=0.1, n_iters=2000)

# Plot the loss against iteration number (the "learning curve")
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Gradient descent: loss vs iteration")
plt.yscale("log")                              # log scale so the late, tiny improvements are visible
plt.show()

print("Final training MSE:", round(losses[-1], 5))
print("Max |w_gd - w_normal|:", np.abs(w_gd - w_normal).max())

**What the plot tells you:** the loss falls steeply at first (big slope = big steps) then flattens as we approach the bottom of the bowl (slope near 0 = tiny steps). Flat means converged.

### 3.4 Effect of the learning rate
Same algorithm, three step sizes. Watch how the choice of $\eta$ changes the story.

In [ ]:
# --- Compare learning rates ---
for lr in [0.001, 0.05, 0.3]:
    _, l = gradient_descent(X_train_s, y_train, lr=lr, n_iters=300)   # fewer iterations for a quick look
    plt.plot(l, label=f"lr = {lr}")

plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Learning rate matters")
plt.yscale("log")
plt.legend()
plt.show()

**What it tells you:** the tiny rate (0.001) is still crawling after 300 steps, while the larger rates reach the bottom fast. If the rate were too big (try `lr=1.5`), the steps overshoot and the loss *increases* or explodes. Try it yourself.

### 3.5 Verify against scikit-learn
`LinearRegression` is scikit-learn's ready-made solver. If our code is right, all three should produce the same weights and the same test predictions.

In [ ]:
# --- Verify our implementations against sklearn ---
from sklearn.linear_model import LinearRegression

# Fit sklearn on the same standardized data (it handles the bias itself via intercept_)
sk = LinearRegression().fit(X_train_s, y_train)
w_sklearn = np.concatenate([[sk.intercept_], sk.coef_])    # same layout as ours: [bias, weights...]

# Compare the weights
print("Max |normal eq - sklearn|     :", np.abs(w_normal - w_sklearn).max())
print("Max |grad descent - sklearn|  :", np.abs(w_gd - w_sklearn).max())

# Compare predictions on the unseen test set
Xb_test = add_bias_column(X_test_s)
pred_normal = Xb_test @ w_normal
pred_gd = Xb_test @ w_gd
pred_sk = sk.predict(X_test_s)

print("Max prediction gap (normal vs sklearn):", np.abs(pred_normal - pred_sk).max())
print("Max prediction gap (GD vs sklearn)    :", np.abs(pred_gd - pred_sk).max())

# np.allclose is True when all numbers match within a tiny tolerance
print("Normal equation matches sklearn:", np.allclose(w_normal, w_sklearn))
print("Gradient descent matches sklearn:", np.allclose(w_gd, w_sklearn, atol=1e-3))

# A first look at test performance: R^2 (1.0 = perfect, 0 = no better than predicting the mean)
ss_res = np.sum((y_test - pred_normal) ** 2)         # residual sum of squares
ss_tot = np.sum((y_test - y_test.mean()) ** 2)       # total variation around the mean
print("Test R^2 (from scratch):", round(1 - ss_res / ss_tot, 4))

**What it tells you:** the gaps are essentially zero, so the from-scratch code reproduces sklearn exactly. Test $R^2$ is about 0.58. The model explains a bit over half the variation in prices, consistent with the limits we spotted in EDA (capping, location non-linearity). We'll improve our understanding of these limits later.

### 3.6 Step 3 summary
| Method | Idea | Pros | Cons |
|---|---|---|---|
| Normal equation | Solve slope = 0 directly | Exact, no tuning | Slow with huge feature counts; fails if $X^TX$ is singular |
| Gradient descent | Walk downhill iteratively | Scales to big data; basis of deep learning | Needs a learning rate and enough iterations |

**Next up (Step 4):** sklearn `LinearRegression`, Ridge and Lasso, with cross-validation.

## 4. The scikit-learn Way: LinearRegression, Ridge and Lasso

In real projects we use tested libraries instead of hand-written solvers. **scikit-learn** (sklearn) gives a uniform pattern: create a model object, call `.fit(X, y)` to train, `.predict(X)` to predict.

### 4.1 Plain LinearRegression
This is the ordinary least squares (OLS) fit from Step 3, now in one line. We'll also compute our first test-set scores. (Metrics get a full treatment in Step 7.)

In [ ]:
# --- Imports for sklearn modeling ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Helper: print test RMSE and R^2 for any fitted model
def report(name, model, Xte, yte):
    pred = model.predict(Xte)                              # predictions on unseen data
    rmse = np.sqrt(mean_squared_error(yte, pred))          # RMSE: typical error in $100k units
    r2 = r2_score(yte, pred)                               # R^2: share of variation explained
    print(f"{name:22s} test RMSE = {rmse:.4f}   test R^2 = {r2:.4f}")

# Fit plain OLS on the standardized training data
ols = LinearRegression().fit(X_train_s, y_train)
report("LinearRegression (OLS)", ols, X_test_s, y_test)

### 4.2 Regularization: why and how
**Overfitting** means a model memorizes quirks (noise) in the training data and then does worse on new data. Linear models overfit when there are many features, few rows, or highly correlated features: the weights can become huge and unstable to chase noise.

**Regularization** adds a *penalty for large weights* to the cost function, so the model must justify every bit of weight it uses:

| Model | Cost function | Penalty on weights | Effect |
|---|---|---|---|
| OLS | $\text{MSE}$ | none | can overfit |
| **Ridge** (L2) | $\text{MSE} + \alpha\sum w_j^2$ | squared size | shrinks all weights smoothly toward 0, rarely exactly 0 |
| **Lasso** (L1) | $\text{MSE} + \alpha\sum \lvert w_j\rvert$ | absolute size | shrinks and sets some weights to **exactly 0** (automatic feature selection) |

- $\alpha$ ("alpha") is the **hyperparameter** (a setting *you* choose, not learned from data) controlling the penalty strength. $\alpha = 0$ gives OLS. A huge $\alpha$ forces all weights to 0 (underfit: too simple).
- The bias $b$ is not penalized.
- This is why we standardized: the penalty treats all weights equally, so features must be on the same scale.
- The trade-off is **bias vs variance**: regularization adds a little bias (systematic error) to reduce variance (sensitivity to the particular training sample).

### 4.3 Choosing alpha with cross-validation
**Cross-validation (CV)** estimates how well a setting generalizes without touching the test set. With 5-fold CV: split the training data into 5 parts (**folds**); train on 4, validate on the 1 left out; rotate so each fold is the validation set once; average the 5 scores. We try many alphas and pick the one with the best average validation score.

In [ ]:
# --- Ridge: cross-validation curve over many alphas ---
alphas = np.logspace(-3, 3, 30)        # 30 values from 0.001 to 1000, evenly spaced on a log scale

cv_mse = []                             # average validation MSE for each alpha
for a in alphas:
    # cross_val_score returns NEGATIVE MSE (sklearn convention: higher score = better), so flip the sign
    scores = cross_val_score(Ridge(alpha=a), X_train_s, y_train, cv=5, scoring="neg_mean_squared_error")
    cv_mse.append(-scores.mean())

plt.semilogx(alphas, cv_mse, marker="o")                      # log x-axis because alpha spans orders of magnitude
plt.axvline(alphas[np.argmin(cv_mse)], color="red", linestyle="--", label="best alpha")
plt.xlabel("alpha (log scale)")
plt.ylabel("5-fold CV MSE")
plt.title("Ridge: choosing alpha by cross-validation")
plt.legend()
plt.show()

# RidgeCV does the same search automatically and refits on all training data with the winner
ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train_s, y_train)
print("Best Ridge alpha:", round(ridge.alpha_, 4))
report("Ridge (CV alpha)", ridge, X_test_s, y_test)

In [ ]:
# --- Lasso: LassoCV searches its own alpha path with 5-fold CV ---
lasso = LassoCV(alphas=np.logspace(-5, 0, 40), cv=5, max_iter=20000).fit(X_train_s, y_train)
print("Best Lasso alpha:", round(lasso.alpha_, 6))
report("Lasso (CV alpha)", lasso, X_test_s, y_test)

# Plot the CV error curve: mse_path_ has shape (n_alphas, n_folds)
plt.semilogx(lasso.alphas_, lasso.mse_path_.mean(axis=1), marker="o")   # mean over the 5 folds
plt.axvline(lasso.alpha_, color="red", linestyle="--", label="best alpha")
plt.xlabel("alpha (log scale)")
plt.ylabel("5-fold CV MSE")
plt.title("Lasso: choosing alpha by cross-validation")
plt.legend()
plt.show()

**What these plots tell you:** the CV error is lowest at a *small* alpha, then rises as alpha grows (too much shrinkage = underfitting). The optimum is small because we have 16,000 training rows and only 8 features, which is a setting where plain OLS is already stable.

### 4.4 Coefficient shrinkage plots
A **coefficient path** plot shows each weight as alpha increases. This is the clearest picture of what regularization does.

In [ ]:
# --- Coefficient paths: fit Ridge and Lasso over a range of alphas and record the weights ---
ridge_alphas = np.logspace(-2, 4, 40)
lasso_alphas = np.logspace(-4, 0.5, 40)

ridge_coefs = [Ridge(alpha=a).fit(X_train_s, y_train).coef_ for a in ridge_alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=20000).fit(X_train_s, y_train).coef_ for a in lasso_alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, alph, coefs, title in [(axes[0], ridge_alphas, np.array(ridge_coefs), "Ridge (L2): smooth shrinkage"),
                               (axes[1], lasso_alphas, np.array(lasso_coefs), "Lasso (L1): weights hit exactly zero")]:
    for j, name in enumerate(feature_names):
        ax.semilogx(alph, coefs[:, j], label=name)       # one line per feature
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("alpha (log scale)")
    ax.set_ylabel("coefficient value")
    ax.set_title(title)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**What the plots tell you:**
- **Ridge:** all lines glide smoothly toward 0 as alpha grows, but none reach it exactly. Correlated features (like `Latitude`/`Longitude`) can shrink at different speeds or even swap direction.
- **Lasso:** lines drop to *exactly* zero one by one. The features that survive the longest (here `MedInc`) are the most useful. This is automatic feature selection.

In [ ]:
# --- Side-by-side coefficient table: OLS vs Ridge vs Lasso (CV-tuned) ---
coef_table = pd.DataFrame({
    "OLS": ols.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_,
}, index=feature_names)
print(coef_table.round(4))

### 4.5 When does regularization actually help? A demo
On this dataset the three models score almost identically, which is an honest and important lesson: **regularization is not magic**. It helps when overfitting is a real risk. Let's create that situation on purpose: use only 150 training rows and add 60 columns of pure random noise (features with no relationship to price).

In [ ]:
# --- Demo: many junk features + few rows = overfitting danger ---
rng = np.random.default_rng(0)                         # reproducible random generator

n_small = 150                                           # tiny training set
noise_train = rng.normal(size=(n_small, 60))            # 60 junk features for training rows
noise_test = rng.normal(size=(X_test_s.shape[0], 60))   # 60 junk features for test rows

# Real 8 features + 60 junk features
Xtr_big = np.hstack([X_train_s[:n_small], noise_train])
ytr_small = y_train[:n_small]
Xte_big = np.hstack([X_test_s, noise_test])

ols_big = LinearRegression().fit(Xtr_big, ytr_small)
ridge_big = RidgeCV(alphas=np.logspace(-1, 4, 40), cv=5).fit(Xtr_big, ytr_small)
lasso_big = LassoCV(alphas=np.logspace(-4, 0, 40), cv=5, max_iter=50000).fit(Xtr_big, ytr_small)

report("OLS  (68 features)", ols_big, Xte_big, y_test)
report("Ridge (68 features)", ridge_big, Xte_big, y_test)
report("Lasso (68 features)", lasso_big, Xte_big, y_test)

# How many of the 60 junk features did Lasso zero out?
junk_zero = np.sum(lasso_big.coef_[8:] == 0)
print(f"\nLasso set {junk_zero} of the 60 junk features exactly to 0")

**What it tells you:** plain OLS does clearly worse here because it uses the junk columns to memorize noise in 150 rows. Ridge and Lasso recover much of the lost accuracy, and Lasso additionally discards most junk features. That is regularization doing its job.

### 4.6 Step 4 summary
| Concept | Takeaway |
|---|---|
| sklearn pattern | `fit` then `predict`; the same results as our scratch code |
| Regularization | Penalize big weights to reduce overfitting |
| Ridge vs Lasso | Ridge shrinks smoothly; Lasso also zeros features out |
| alpha | Hyperparameter chosen by cross-validation, never by the test set |
| Reality check | With plenty of data and few features, OLS is already fine |

**Next up (Step 5):** assumption checks (residuals, Q-Q, heteroscedasticity, VIF).